# Section 7.1 Runtime Composition Visualizations

This notebook reads the detailed timing summary CSV files from `exp1` through `exp4` and visualizes both:

- total algorithm runtime across `Non-random`, `Random Sampling`, and `Random Projection`
- runtime composition inside each method using actual stacked runtime bars

Design choice:

- the top subplot compares total runtime across methods
- the bottom three subplots show actual stacked runtime bars for each method
- each stack segment keeps its real runtime height in seconds
- percentage labels are written inside the segments when large enough to read

This is intended to make it easier to report both absolute runtime and relative contribution at the same time.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"
OUTPUT_ROOT = EXPERIMENTS_ROOT / "timing_percentages"

METHOD_ORDER = ["Non-random", "Random Sampling", "Random Projection"]
METHOD_RUNTIME_COLORS = {
    "Non-random": "#2ca02c",
    "Random Sampling": "#ff7f0e",
    "Random Projection": "#1f77b4",
}

EXPERIMENT_CONFIG = {
    "exp1": {
        "label": "Exp1 (n variation)",
        "summary_filename": "exp1_timing_breakdown_summary.csv",
        "x_col": "n",
        "x_label": "n",
    },
    "exp2": {
        "label": "Exp2 (alpha_n variation)",
        "summary_filename": "exp2_timing_breakdown_summary.csv",
        "x_col": "alpha_n",
        "x_label": "alpha_n",
    },
    "exp3": {
        "label": "Exp3 (K variation)",
        "summary_filename": "exp3_timing_breakdown_summary.csv",
        "x_col": "K",
        "x_label": "K",
    },
    "exp4": {
        "label": "Exp4 (n variation, alpha_n = 2/sqrt(n))",
        "summary_filename": "exp4_timing_breakdown_summary.csv",
        "x_col": "n",
        "x_label": "n",
    },
}

METHOD_COMPONENTS = {
    "Non-random": [
        ("nr_eig_sec", "Spectral decomposition on original matrix A"),
        ("nr_kmeans_sec", "K-means on spectral embedding"),
    ],
    "Random Sampling": [
        ("rs_sample_mask_sec", "Generate Bernoulli sampling mask"),
        ("rs_build_sampled_matrix_sec", "Construct sampled matrix A_s"),
        ("rs_eig_sec", "Eigen decomposition on sampled A_s"),
        ("rs_reconstruct_sec", "Reconstruct low-rank A_hat"),
        ("rs_symmetrize_sec", "Symmetrize reconstructed A_hat"),
        ("rs_kmeans_sec", "K-means on sampled spectral embedding"),
    ],
    "Random Projection": [
        ("rp_draw_omega_sec", "Generate Gaussian test matrix Omega"),
        ("rp_power_iter_sec", "Power iterations with A"),
        ("rp_qr_sec", "Construct QR basis Q"),
        ("rp_build_core_sec", "Project A to core matrix C"),
        ("rp_reconstruct_sec", "Reconstruct low-rank A_hat"),
        ("rp_small_eig_sec", "Eigen decomposition on core matrix C"),
        ("rp_lift_sec", "Lift embedding back with Q"),
        ("rp_kmeans_sec", "K-means on projected embedding"),
    ],
}

METHOD_PALETTES = {
    "Non-random": ["#117733", "#CC6677", "#BBBBBB"],
    "Random Sampling": ["#4477AA", "#EE6677", "#228833", "#CCBB44", "#66CCEE", "#AA3377", "#BBBBBB"],
    "Random Projection": ["#332288", "#88CCEE", "#44AA99", "#117733", "#999933", "#DDCC77", "#CC6677", "#AA4499", "#BBBBBB"],
}


In [ ]:
def find_latest_summary(summary_filename):
    matches = list(EXPERIMENTS_ROOT.glob(f"**/{summary_filename}"))
    if not matches:
        return None
    return max(matches, key=lambda path: path.stat().st_mtime)


def load_summary(exp_key):
    cfg = EXPERIMENT_CONFIG[exp_key]
    summary_path = find_latest_summary(cfg["summary_filename"])
    if summary_path is None:
        return None, None
    return summary_path, pd.read_csv(summary_path)


def numeric_series(df, column_name):
    if column_name not in df.columns:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    return pd.to_numeric(df[column_name], errors="coerce").fillna(0.0)


def format_x_labels(values, x_col):
    labels = []
    for value in values:
        if x_col == "alpha_n":
            labels.append(f"{float(value):.2f}")
        else:
            try:
                labels.append(str(int(float(value))))
            except Exception:
                labels.append(str(value))
    return labels


def build_method_runtime_table(df, method_name, x_col):
    method_df = df[df["method"] == method_name].copy()
    if method_df.empty:
        return None

    method_df[x_col] = pd.to_numeric(method_df[x_col], errors="coerce")
    method_df = method_df.sort_values(x_col).reset_index(drop=True)
    total = numeric_series(method_df, "algo_total_sec_mean")
    if (total <= 0).all():
        total = numeric_series(method_df, "time_sec_mean")

    result = pd.DataFrame({x_col: method_df[x_col], "algo_total_sec_mean": total})
    component_sum = pd.Series(np.zeros(len(method_df)), index=method_df.index, dtype=float)

    for metric_name, label in METHOD_COMPONENTS[method_name]:
        values = numeric_series(method_df, f"{metric_name}_mean")
        result[label] = values
        component_sum = component_sum + values

    result["Measured overhead / remainder"] = (total - component_sum).clip(lower=0.0)

    for col in [c for c in result.columns if c not in {x_col, "algo_total_sec_mean"}]:
        pct = np.where(total > 0, 100.0 * result[col] / total, 0.0)
        result[f"{col} (%)"] = pct

    return result


def display_runtime_table(exp_label, method_name, table):
    display(Markdown(f"#### {exp_label} | {method_name} runtime table"))
    display(table.round(4))


def plot_total_runtime_comparison(ax, df, x_col, x_label):
    for method_name in METHOD_ORDER:
        method_df = df[df["method"] == method_name].copy()
        if method_df.empty:
            continue
        method_df[x_col] = pd.to_numeric(method_df[x_col], errors="coerce")
        method_df = method_df.sort_values(x_col)
        y = numeric_series(method_df, "algo_total_sec_mean")
        if (y <= 0).all():
            y = numeric_series(method_df, "time_sec_mean")
        ax.plot(
            method_df[x_col].to_numpy(dtype=float),
            y.to_numpy(dtype=float),
            color=METHOD_RUNTIME_COLORS[method_name],
            linewidth=2.4,
            marker="o",
            label=method_name,
        )
    ax.set_title("Total runtime comparison")
    ax.set_xlabel(x_label)
    ax.set_ylabel("Algorithm runtime (sec)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", frameon=False)


def annotate_stack_percentages(ax, x_pos, bottoms, heights, percentages, shared_ymax):
    for xpos, bottom, height, pct in zip(x_pos, bottoms, heights, percentages):
        if height <= 0:
            continue
        if pct < 8.0:
            continue
        if height < shared_ymax * 0.06:
            continue
        y = bottom + height / 2.0
        ax.text(
            xpos,
            y,
            f"{pct:.0f}%",
            ha="center",
            va="center",
            fontsize=7,
            color="black",
            bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none", "alpha": 0.8},
        )


def plot_method_runtime_stack(ax, table, method_name, x_col, x_label, shared_ymax):
    component_cols = [col for col in table.columns if col not in {x_col, "algo_total_sec_mean"} and not col.endswith(" (%)")]
    colors = METHOD_PALETTES[method_name][: len(component_cols)]
    x_labels = format_x_labels(table[x_col].tolist(), x_col)
    x_pos = np.arange(len(x_labels))
    bottom = np.zeros(len(x_labels), dtype=float)

    for color, col in zip(colors, component_cols):
        values = table[col].to_numpy(dtype=float)
        pct_values = table[f"{col} (%)"].to_numpy(dtype=float)
        hatch = "//" if col == "Measured overhead / remainder" else None
        edgecolor = "#666666" if col == "Measured overhead / remainder" else "white"
        ax.bar(
            x_pos,
            values,
            bottom=bottom,
            color=color,
            edgecolor=edgecolor,
            linewidth=0.8,
            hatch=hatch,
            label=col,
        )
        annotate_stack_percentages(ax, x_pos, bottom, values, pct_values, shared_ymax)
        bottom = bottom + values

    ax.set_ylim(0, shared_ymax)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel("Runtime (sec)")
    ax.set_xlabel(x_label)
    ax.set_title(method_name)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
        title=f"{method_name} steps",
    )


def render_runtime_composition(exp_key):
    cfg = EXPERIMENT_CONFIG[exp_key]
    summary_path, df = load_summary(exp_key)
    display(Markdown(f"## {cfg['label']}"))

    if summary_path is None or df is None:
        print(
            "Timing summary CSV not found. "
            f"Expected filename: {cfg['summary_filename']}."
        )
        return

    print(f"Loaded summary CSV: {summary_path}")
    out_dir = OUTPUT_ROOT / exp_key
    out_dir.mkdir(parents=True, exist_ok=True)

    tables = {}
    max_total = 0.0
    for method_name in METHOD_ORDER:
        table = build_method_runtime_table(df, method_name, cfg["x_col"])
        tables[method_name] = table
        if table is not None:
            display_runtime_table(cfg["label"], method_name, table)
            max_total = max(max_total, float(table["algo_total_sec_mean"].max()))

    shared_ymax = max_total * 1.10 if max_total > 0 else 1.0

    fig, axes = plt.subplots(
        4,
        1,
        figsize=(15, 18),
        gridspec_kw={"height_ratios": [1.2, 1.0, 1.0, 1.0]},
    )

    plot_total_runtime_comparison(axes[0], df, cfg["x_col"], cfg["x_label"])

    for ax, method_name in zip(axes[1:], METHOD_ORDER):
        table = tables[method_name]
        if table is None:
            ax.set_visible(False)
            continue
        plot_method_runtime_stack(ax, table, method_name, cfg["x_col"], cfg["x_label"], shared_ymax)

    fig.suptitle(f"{cfg['label']} | Runtime comparison and composition", fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 0.72, 0.98])

    out_png = out_dir / f"{exp_key}_runtime_composition.png"
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved figure: {out_png}")


In [ ]:
render_runtime_composition("exp1")


In [ ]:
render_runtime_composition("exp2")


In [ ]:
render_runtime_composition("exp3")


In [ ]:
render_runtime_composition("exp4")
